# 🚀 Upper Bound: Joint Training (Multitask Benchmark)
### Continual Relation Extraction on FewRel Track A (80 relations, 5-shot, seed 2021)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/main/notebooks/upper_bound_joint.ipynb)

Notebook này triển khai thực nghiệm **Cận trên (Upper Bound — Joint Training)**:
- Huấn luyện đa nhiệm cùng lúc toàn bộ 80 quan hệ (400 mẫu train, 15 epochs).
- Không phân tách thời gian, không có hiện tượng quên lãng ($AF = 0$).
- Thiết lập trần hiệu năng ($AA \approx 52.12\%$) để đo đạc khoảng cách tối ưu hóa so với Baseline B0.


## 1. Kiểm tra GPU Preflight (CUDA Verification)


In [ ]:
import torch

print('=' * 60)
print('KIỂM TRA THIẾT BỊ TÍNH TOÁN (GPU PREFLIGHT)')
print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ CUDA Khả dụng: True')
    print(f'✓ GPU Thiết bị: {torch.cuda.get_device_name(0)}')
    print(f'✓ Bộ nhớ VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    !nvidia-smi
else:
    print('⚠️ CẢNH BÁO: CUDA KHÔNG KHẢ DỤNG! Bạn đang chạy trên CPU.')
    print('👉 Hướng dẫn bật GPU: Chọn Runtime trên thanh menu -> Change runtime type -> Chọn T4 GPU -> Lưu.')


## 2. Thiết lập Codebase & Cài đặt Thư viện


In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
REPO_NAME = 'research-code-lab'

if not Path(REPO_NAME).exists():
    print(f'Đang clone repository từ {REPO_URL}...')
    !git clone {REPO_URL}

%cd {REPO_NAME}

# Cài đặt các thư viện cần thiết
!pip install -q transformers torch matplotlib pandas


## 3. Tự động tải Dữ liệu FewRel thô (Zero-Manual Setup)


In [ ]:
import urllib.request
from pathlib import Path

data_dir = Path('dataset-pipelines/continual-relation-extraction/data/raw/fewrel')
data_dir.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://raw.githubusercontent.com/thunlp/FewRel/master/data'
FILES = ['train_wiki.json', 'val_wiki.json', 'pid2name.json']

for fname in FILES:
    fpath = data_dir / fname
    if not fpath.exists():
        print(f'Đang tải {fname} từ {BASE_URL}/{fname}...')
        urllib.request.urlretrieve(f'{BASE_URL}/{fname}', fpath)
        print(f'  ✓ Hoàn tất {fname} ({fpath.stat().st_size / 1024:.1f} KB)')
    else:
        print(f'  ✓ Đã có sẵn {fname} ({fpath.stat().st_size / 1024:.1f} KB)')

print('Dữ liệu FewRel đã sẵn sàng 100%!')


## 4. Thực thi Huấn luyện Upper Bound (Joint Training)


In [ ]:
# Huấn luyện đa nhiệm 80 lớp và chọn checkpoint tốt nhất trên tập Validation
!python experiments/run_joint_training.py --device auto


## 5. Trực quan hóa & So sánh Đối đầu (Upper Bound vs B0 Lower Bound)


In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

results_dir = Path('results/fewrel/5shot/Upper_Bound_joint/seed_2021')
summary_path = results_dir / 'summary.json'
csv_path = results_dir / 'per_task_breakdown.csv'

if summary_path.exists():
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    print('=' * 65)
    print('UPPER BOUND JOINT TRAINING — KẾT QUẢ THỰC NGHIỆM')
    print('=' * 65)
    print(f'Overall 80-Way Test Accuracy: {summary["overall_test_accuracy"] * 100:.2f}%')
    print(f'Overall 80-Way Test Macro-F1: {summary["overall_test_macro_f1"] * 100:.2f}%')
    print(f'Best Val Accuracy (Epoch {summary["best_val_epoch"]}): {summary["best_val_accuracy"] * 100:.2f}%')
    
    b0_comp = summary.get('b0_comparison', {})
    if b0_comp:
        print(f'\nSo sánh Đối đầu với Lower Bound (B0):')
        print(f'  B0 Final Accuracy:          {b0_comp["b0_final_accuracy"] * 100:.2f}%')
        print(f'  Khoảng cách trần (Gain):    +{b0_comp["accuracy_gain_over_b0"] * 100:.2f}%')
        print(f'  Macro-F1 Gain:              +{b0_comp["macro_f1_gain_over_b0"] * 100:.2f}%')
    print('=' * 65)

if csv_path.exists():
    print('\n📊 Bảng phân tích chi tiết hiệu năng theo từng Task (1,400 test mẫu/task):')
    df = pd.read_csv(csv_path)
    display(df)


## 6. Tải Artifacts về Máy tính Cá nhân (Optional)


In [ ]:
import shutil

try:
    from google.colab import files
    zip_name = 'Upper_Bound_joint_results'
    print('Đang nén kết quả thành file zip...')
    shutil.make_archive(zip_name, 'zip', results_dir)
    files.download(f'{zip_name}.zip')
    print(f'✓ Đã bắt đầu tải xuống {zip_name}.zip')
except ImportError:
    print('Chức năng tải file tự động chỉ hoạt động trên Google Colab.')
